# NB-2 — T1, the leakage ladder

The headline table. Each rung closes one more leakage channel with the model held
constant. Attach `halo-stage1`.

Watch the rung 2 → rung 3 step: that is entity leakage, and it is the paper.

**Publish output as `halo-stage2`.**


In [ ]:
# --- HALO bootstrap -------------------------------------------------------------
# Attach these datasets to this notebook before running (right panel -> Add Data):
#   1. Competition: "ieee-fraud-detection"      (accept the rules first)
#   2. Your source dataset: "halo-src"           (the halo/ package, see RUN_GUIDE.md)
#   3. For stages after NB-1: the previous stage's output dataset (e.g. halo-stage8)
import os, shutil, sys, subprocess, time

from pathlib import Path

# 1. Locate and copy halo package regardless of Kaggle mount structure or folder naming
if os.path.exists("/kaggle/working/halo"):
    shutil.rmtree("/kaggle/working/halo")

HALO_FOUND = False
candidates = []
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "working" in root or "__pycache__" in root:
            continue
        if "cli.py" in files and "config.py" in files:
            score = 0
            if "figures.py" in files:
                score += 20  # Prioritize latest code containing figures.py
            if "__init__.py" in files:
                score += 5
            if "halo" in os.path.basename(root).lower():
                score += 2
            candidates.append((score, root))

if candidates:
    candidates.sort(key=lambda x: x[0], reverse=True)
    best_src = candidates[0][1]
    shutil.copytree(best_src, "/kaggle/working/halo", dirs_exist_ok=True)
    init_f = os.path.join("/kaggle/working/halo", "__init__.py")
    if not os.path.exists(init_f):
        with open(init_f, "w") as f:
            f.write("# HALO package root
")
    HALO_FOUND = True
    print(f"Loaded HALO package from: {best_src}")
else:
    # Also check for any zip archives
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".zip") and ("halo" in f.lower() or "src" in f.lower()):
                import zipfile
                zf_path = os.path.join(root, f)
                try:
                    with zipfile.ZipFile(zf_path, "r") as zf:
                        zf.extractall("/kaggle/working")
                    if os.path.exists("/kaggle/working/halo"):
                        HALO_FOUND = True
                        print(f"Extracted halo from {zf_path}")
                        break
                except Exception:
                    pass
        if HALO_FOUND:
            break

if not HALO_FOUND:
    raise FileNotFoundError("Could not find HALO package files under /kaggle/input! Please attach 'halo-src' dataset.")

sys.path.insert(0, "/kaggle/working")

# 2. Locate IEEE-CIS competition data
IEEE_CANDIDATES = [
    "/kaggle/input/competitions/ieee-fraud-detection",
    "/kaggle/input/ieee-fraud-detection",
]
IEEE_DIR = next((p for p in IEEE_CANDIDATES if os.path.exists(p)), None)
if IEEE_DIR is None and os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "train_transaction.csv" in files:
            IEEE_DIR = root
            break

if IEEE_DIR:
    try:
        import halo.config as _cfg
        _cfg.IEEE_DIR = Path(IEEE_DIR)
        print(f"Using IEEE_DIR: {_cfg.IEEE_DIR}")
    except Exception as e:
        pass

# 3. Carry forward checkpoints, results, and figures from any previous halo-stage
if os.path.exists("/kaggle/input"):
    copied_count = 0
    # First extract any bundled results zip from previous stages if present
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".zip") and ("halo_results" in f.lower() or "results" in f.lower()):
                import zipfile
                zf_path = os.path.join(root, f)
                try:
                    with zipfile.ZipFile(zf_path, "r") as zf:
                        zf.extractall("/kaggle/working")
                    print(f"Unpacked previous stage bundle: {f}")
                except Exception:
                    pass

    for root, dirs, files in os.walk("/kaggle/input"):
        base = os.path.basename(root)
        if base in ("checkpoints", "results", "figures") and "working" not in root:
            dest = os.path.join("/kaggle/working", base)
            os.makedirs(dest, exist_ok=True)
            for f in files:
                shutil.copy2(os.path.join(root, f), os.path.join(dest, f))
                copied_count += 1
    if copied_count > 0:
        print(f"Carried forward {copied_count} files from previous stages.")

# 4. Locate PaySim dataset (for NB-7)
PAYSIM_CANDIDATES = [
    "/kaggle/input/datasets/ealaxi/paysim1/PS_20174392719_1491204439457_log.csv",
    "/kaggle/input/paysim1/PS_20174392719_1491204439457_log.csv",
    "/kaggle/input/competitions/paysim1/PS_20174392719_1491204439457_log.csv",
]
PAYSIM_CSV = next((p for p in PAYSIM_CANDIDATES if os.path.exists(p)), None)
if PAYSIM_CSV is None and os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.startswith("PS_") and f.endswith(".csv"):
                PAYSIM_CSV = os.path.join(root, f)
                break
        if PAYSIM_CSV:
            break

if PAYSIM_CSV:
    try:
        import halo.config as _cfg
        _cfg.PAYSIM_CSV = Path(PAYSIM_CSV)
        print(f"Using PAYSIM_CSV: {_cfg.PAYSIM_CSV}")
    except Exception as e:
        pass

from halo.io import environment_manifest
env = environment_manifest()
print("ENVIRONMENT (observed, not assumed):")
for k, v in env.items():
    print(f"  {k:24s} {v}")


In [ ]:
from halo.cli import main
main(["run-ladder", "--seeds", "0", "1", "2", "3", "4"])
